# ARC-AGI-3 Solo — Submission v0 (random baseline)

Sanity-check submission. Plays every game in the competition dataset with a uniformly random agent. Goal: confirm the plumbing (offline `arc_agi` engine in COMPETITION mode, scorecard generation, Kaggle submission file) is correct end-to-end. Score will be near zero — that's expected.

**Self-contained:** no internet, no external imports beyond what's in the comp dataset's `arc_agi_3_wheels/`.

## 1. Install vendored wheels

In [ ]:
import subprocess, sys, glob, os

WHEEL_DIR = '/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, '*.whl')))
print(f'Found {len(wheels)} wheels in {WHEEL_DIR}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
    '--find-links', WHEEL_DIR, 'arc-agi', 'arcengine',
])
print('install ok')

## 2. Force COMPETITION (offline) mode

On Kaggle there's no internet. `arc_agi.Arcade` reads `OPERATION_MODE` and `ENVIRONMENTS_DIR` from env.

In [ ]:
os.environ['OPERATION_MODE'] = 'competition'
os.environ['ENVIRONMENTS_DIR'] = '/kaggle/input/arc-prize-2026-arc-agi-3/environment_files'

from arc_agi import Arcade
from arcengine import FrameData, FrameDataRaw, GameAction, GameState
print('arc_agi imported, mode =', os.environ['OPERATION_MODE'])

## 3. Minimal Agent base class

Same shape as the vendor `Agent` (`is_done` + `choose_action`), but no langgraph/smolagents/openai dep tree.

In [ ]:
import logging, random, time
from abc import ABC, abstractmethod
from typing import Optional
from arc_agi import EnvironmentWrapper

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
log = logging.getLogger()

class Agent(ABC):
    MAX_ACTIONS = 5000
    def __init__(self, card_id, game_id, agent_name, arc_env, tags=None):
        self.card_id = card_id
        self.game_id = game_id
        self.agent_name = agent_name
        self.arc_env = arc_env
        self.tags = tags or []
        self.frames = [FrameData(levels_completed=0)]
        self.action_counter = 0
        self.timer = 0.0
    @property
    def name(self): return f'{self.game_id}.{self.__class__.__name__.lower()}'
    @property
    def fps(self):
        if self.action_counter == 0: return 0.0
        return round(self.action_counter / max(time.time()-self.timer, 0.1), 2)
    def main(self):
        self.timer = time.time()
        while not self.is_done(self.frames, self.frames[-1]) and self.action_counter <= self.MAX_ACTIONS:
            action = self.choose_action(self.frames, self.frames[-1])
            frame = self._step(action)
            if frame is not None:
                self.frames.append(frame)
            self.action_counter += 1
    def _step(self, action):
        try:
            data = action.action_data.model_dump()
            raw = self.arc_env.step(action, data=data, reasoning=data.get('reasoning', {}))
        except Exception:
            log.exception('step failed for %s', action.name)
            return None
        if raw is None: return None
        return FrameData(
            game_id=raw.game_id, frame=[a.tolist() for a in raw.frame], state=raw.state,
            levels_completed=raw.levels_completed, win_levels=raw.win_levels,
            guid=raw.guid, full_reset=raw.full_reset, available_actions=raw.available_actions,
        )
    @abstractmethod
    def is_done(self, frames, latest_frame): ...
    @abstractmethod
    def choose_action(self, frames, latest_frame): ...

## 4. Random agent

In [ ]:
class RandomAgent(Agent):
    MAX_ACTIONS = 5000
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        random.seed(int(time.time() * 1e6) ^ (hash(self.game_id) & 0xFFFFFFFF))
    def is_done(self, frames, latest_frame):
        return latest_frame.state is GameState.WIN
    def choose_action(self, frames, latest_frame):
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        avail = latest_frame.available_actions or [a.value for a in GameAction if a is not GameAction.RESET]
        avail = [a for a in avail if a != 0]
        if not avail:
            return GameAction.RESET
        action = GameAction.from_id(random.choice(avail))
        if action.is_complex():
            action.set_data({'x': random.randint(0, 63), 'y': random.randint(0, 63)})
            action.reasoning = {'desired_action': action.value, 'my_reason': 'random'}
        else:
            action.reasoning = 'random'
        return action

## 5. Discover games

In [ ]:
from pathlib import Path
ENV_DIR = Path(os.environ['ENVIRONMENTS_DIR'])
games = sorted({p.name for p in ENV_DIR.iterdir()
                if p.is_dir() and any(level.is_dir() for level in p.iterdir())})
print(f'Discovered {len(games)} games:', games)

## 6. Run

Sequential per-game. Kaggle notebook limit is 9h on CPU/12h on GPU; budget 1500 actions per game ≈ 7-8 min/game (~3h for 25 games). Tune `MAX_ACTIONS` if anything starts thrashing.

In [ ]:
import json
RandomAgent.MAX_ACTIONS = 1500

arc = Arcade()
card_id = arc.open_scorecard(tags=['random', 'v0'])
log.info(f'scorecard {card_id}, mode {arc.operation_mode}')

t0 = time.time()
for gid in games:
    try:
        env = arc.make(gid, scorecard_id=card_id)
        if env is None:
            log.warning('env is None for %s; skipping', gid); continue
        agent = RandomAgent(card_id=card_id, game_id=gid, agent_name='random', arc_env=env, tags=['random', 'v0'])
        agent.main()
        log.info(f'{gid}: levels {agent.frames[-1].levels_completed}, actions {agent.action_counter}, total {time.time()-t0:.0f}s')
    except Exception:
        log.exception(f'{gid} crashed; continuing')

scorecard = arc.close_scorecard(card_id)
dumped = scorecard.model_dump()
print(f"\n=== Final: {dumped['total_levels_completed']}/{dumped['total_levels']} levels, {dumped['total_environments_completed']}/{dumped['total_environments']} envs, {dumped['total_actions']} actions ===")

## 7. Write submission

Scoring is server-side based on the `arc_agi` scorecard. We dump the full JSON to `/kaggle/working/scorecard.json` for posterity and so the Kaggle submission UI has something to attach.

In [ ]:
with open('/kaggle/working/scorecard.json', 'w') as f:
    json.dump(dumped, f, indent=2, default=str)
print('wrote /kaggle/working/scorecard.json')
# Per-environment summary
for env in dumped['environments']:
    run = env['runs'][0] if env['runs'] else {}
    print(f"{env['id']:>22} | levels {env['levels_completed']:>2}/{env['level_count']:>2} | actions {env['actions']:>5} | resets {env['resets']}")